# PROJECT NOIR ? The Nishidak Broadcast
### One-click Colab render: 05:52 workprint with the original score
**Visual direction:** Sayan ? **Soundtrack:** `Noir.wav` by Atandra

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sayan-bhattacharya/noir-analog-horror/blob/main/colab_noir_accelerator.ipynb)

Run **Runtime ? Run all**. The notebook fetches the original `Noir.wav` from this NOIR repository using Git LFS, installs render dependencies, renders the current 05:52 workprint, checks that the MP4 has picture and audio, previews it, and downloads it. If LFS cannot provide the score, it tries your mounted Drive and then prompts for upload. No podcast audio is used.

This is the 05:52 workprint, not the complete 30:22 film. Shot 30 keeps the slow courtyard clouds, restrained analog texture, purple charged-particle portal, and Bimal's slow approach, slip, and fall.


## 1. Runtime check

The renderer is primarily CPU/OpenCV and FFmpeg based; a GPU is optional. `RENDER_PREVIEW` in the render cell controls speed versus full 720p output.


In [ ]:
import torch
if torch.cuda.is_available():
    print(f"[SUCCESS] GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[INFO] Running on Standard CPU. For fast renders, select Runtime -> Change runtime type -> T4 GPU")

## 2. Get the NOIR project, install dependencies, and locate the score


In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

# Git LFS is needed to fetch NOIR's 321 MB original score from this repository.
try:
    subprocess.run(["git", "lfs", "version"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except (subprocess.CalledProcessError, FileNotFoundError):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=True)
subprocess.run(["git", "lfs", "install"], check=True)

project_dir = Path("/content/noir-analog-horror")
if not project_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sayan-bhattacharya/noir-analog-horror.git", str(project_dir)], check=True)
else:
    # Update committed code without resetting the checkout or its generated files.
    subprocess.run(["git", "-C", str(project_dir), "pull", "--ff-only"], check=True)
# Fetch large tracked media explicitly; if unavailable, later cells try Drive/upload.
lfs_result = subprocess.run(["git", "-C", str(project_dir), "lfs", "pull"], text=True, capture_output=True)
if lfs_result.returncode == 0:
    print("Fetched repository LFS assets.")
else:
    print("Git LFS score fetch unavailable; will try Drive or the upload prompt.")
    print(lfs_result.stderr[-1200:])
os.chdir(project_dir)
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
os.environ["PYTHONPATH"] = str(project_dir)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless", "soundfile", "scipy", "pillow", "uharfbuzz", "freetype-py", "pyyaml"], check=True)
if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    raise RuntimeError("Colab's FFmpeg tools are unavailable. Restart with a standard hosted runtime and run all again.")

# Prefer the LFS checkout; retain Drive and upload fallbacks for offline/quota issues.
from google.colab import drive, files
import soundfile as sf

drive_mounted = False
try:
    drive.mount("/content/drive", timeout_ms=60000)
    drive_mounted = True
except Exception as exc:
    print(f"Drive mount unavailable ({type(exc).__name__}). Will use upload if needed.")

score_candidates = [project_dir / "Noir.wav", project_dir / "audio" / "Noir.wav"]
if drive_mounted:
    score_candidates.append(Path("/content/drive/MyDrive/NOIR_Film/Noir.wav"))

def is_valid_noir_score(path):
    try:
        info = sf.info(str(path))
        return info.frames > 0 and info.frames / info.samplerate >= 352
    except Exception:
        return False

score_path = next((candidate for candidate in score_candidates if candidate.is_file() and is_valid_noir_score(candidate)), None)
if score_path is None:
    print("Select the original NOIR soundtrack file named Noir.wav. Do not upload a podcast mix.")
    uploaded = files.upload()
    score_path = next((Path(name) for name in uploaded if Path(name).name.lower() == "noir.wav"), None)
if score_path is None or not score_path.is_file() or not is_valid_noir_score(score_path):
    raise FileNotFoundError("A valid Noir.wav of at least 352 seconds is required. No other audio will be substituted.")

score_info = sf.info(str(score_path))
score_duration = score_info.frames / score_info.samplerate
project_score = project_dir / "Noir.wav"
if score_path.resolve() != project_score.resolve():
    shutil.copy2(score_path, project_score)
print(f"Original NOIR score ready: {project_score.name} | {score_duration:.3f}s | {score_info.samplerate} Hz | {score_info.channels} channel(s)")


## 3. Render the 05:52 workprint

The source score is longer than this workprint. The render uses its opening 352 seconds and does not stretch the picture to the full 30:22 screenplay. Default: full-resolution 1280?720 at 24 fps. Set `RENDER_PREVIEW = True` for a quicker 640?360, 12 fps draft.


In [ ]:
import os, sys, time, subprocess
from pathlib import Path

os.chdir("/content/noir-analog-horror")
os.environ["PYTHONPATH"] = "."
RENDER_SECONDS = 352
RENDER_PREVIEW = False  # True = quicker 640x360 at 12 fps draft
RENDER_QUALITY = "preview" if RENDER_PREVIEW else "final"
cmd = [sys.executable, "-m", "noir_film.render_noir", "--duration", str(RENDER_SECONDS), "--quality", RENDER_QUALITY]
if RENDER_PREVIEW:
    cmd.append("--preview")
print("Rendering NOIR 05:52 workprint:", " ".join(cmd), flush=True)
start_time = time.time()
subprocess.run(cmd, check=True)
print(f"Render complete in {(time.time() - start_time) / 60:.1f} minutes")


## 4. Verify the rendered video and soundtrack


In [ ]:
import glob, json, os, subprocess
from pathlib import Path
from IPython.display import Video, display

matches = glob.glob("/content/noir-analog-horror/output/noir_master_352s_*.mp4")
if not matches:
    raise FileNotFoundError("The 05:52 NOIR render was not found. Check the render cell output.")
video_path = Path(max(matches, key=os.path.getmtime))
probe = subprocess.run([
    "ffprobe", "-v", "error", "-show_entries", "format=duration:stream=codec_type,codec_name,duration",
    "-of", "json", str(video_path)
], check=True, capture_output=True, text=True)
info = json.loads(probe.stdout)
streams = info.get("streams", [])
video_stream = next((s for s in streams if s.get("codec_type") == "video"), None)
audio_stream = next((s for s in streams if s.get("codec_type") == "audio"), None)
container_duration = float(info.get("format", {}).get("duration", 0))
if video_stream is None or audio_stream is None:
    raise RuntimeError("QC failed: output must contain both picture and soundtrack.")
if abs(container_duration - 352.0) > 0.25:
    raise RuntimeError(f"QC failed: expected a 352-second workprint, got {container_duration:.3f}s.")
print(f"QC passed: {video_path.name}")
print(f"Duration: {container_duration:.3f}s | Video: {video_stream.get('codec_name')} | Audio: {audio_stream.get('codec_name')} from Noir.wav")
print(f"File size: {video_path.stat().st_size / (1024**3):.2f} GiB")
display(Video(str(video_path), embed=False, width=900))


## 5. Download the verified NOIR workprint

The downloaded MP4 includes the original `Noir.wav` soundtrack.


In [ ]:
from google.colab import files
from pathlib import Path
import glob, os
matches = glob.glob("/content/noir-analog-horror/output/noir_master_352s_*.mp4")
if not matches:
    raise FileNotFoundError("NOIR render not found. Run the notebook from the top.")
video_path = Path(max(matches, key=os.path.getmtime))
print("Downloading verified NOIR workprint:", video_path.name)
files.download(str(video_path))


## 6. Optional: save another copy to Google Drive

The verified MP4 downloads in the previous cell. This optional cell also copies it to `MyDrive/NOIR_Film/renders`; if Drive is unavailable, it offers a browser download.


In [ ]:
import shutil, glob, os
from pathlib import Path
from google.colab import drive, files
matches = glob.glob("/content/noir-analog-horror/output/noir_master_352s_*.mp4")
if not matches:
    raise FileNotFoundError("NOIR render not found. Run the notebook from the top.")
src = Path(max(matches, key=os.path.getmtime))
try:
    drive.mount("/content/drive", timeout_ms=60000)
    dest_dir = Path("/content/drive/MyDrive/NOIR_Film/renders")
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / src.name
    shutil.copy2(src, dest)
    print(f"Saved NOIR workprint to Drive: {dest}")
except Exception as exc:
    print(f"Drive save unavailable ({type(exc).__name__}); downloading the MP4 directly instead.")
    files.download(str(src))
